# 03 — Submissão: TF-IDF de palavras + caracteres com regressão logística

Notebook de entrega da competição [Name That LLM](https://www.kaggle.com/competitions/name-that-llm). Ele treina **só a configuração vencedora** da busca feita em `02-sanjos-baselines-tfidf.ipynb` e gera `submission.parquet`.

Esse notebook aborda:

  7. Organizar apresentação da solução
    - Modelo final
    - Submissão

## Por que esta configuração

O notebook 02 comparou 5 modelos e testou 128 combinações de vetorizador, função de custo e penalização, com validação cruzada agrupada por prompt (`GroupKFold`, 5 folds):

| Decisão | Evidência no notebook 02 |
|---|---|
| Palavras (1–2) **+** caracteres | A união superou "só caracteres" nos 5 folds |
| Caracteres com `analyzer="char"` | Superou `char_wb` em média e no máximo |
| `lowercase=False` | Ganhou em todas as combinações: o uso de maiúsculas faz parte do estilo |
| Regressão logística com `C=10` | Melhor modelo da busca: **32,52% ± 1,86%** de balanced accuracy (acaso: 5,6%) |

## Setup

In [1]:
from pathlib import Path

import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import FeatureUnion, Pipeline

CAMINHOS_KAGGLE = [Path("/kaggle/input/competitions/name-that-llm"), Path("/kaggle/input/name-that-llm")]
NO_KAGGLE = any(p.exists() for p in CAMINHOS_KAGGLE)
DATA_DIR = next((p for p in CAMINHOS_KAGGLE if p.exists()), Path("../data"))
SAIDA_DIR = Path(".") if NO_KAGGLE else Path("../submissions")

## 7.1 Modelo final

In [2]:
treino = pd.read_csv(DATA_DIR / "train.csv")

modelo = Pipeline([
    ("vect", FeatureUnion([
        ("palavras", TfidfVectorizer(ngram_range=(1, 2), sublinear_tf=True)),
        ("caracteres", TfidfVectorizer(analyzer="char", ngram_range=(2, 5), sublinear_tf=True, lowercase=False)),
    ])),
    ("clf", LogisticRegression(C=10, max_iter=1000)),
])
modelo.fit(treino["text"], treino["model"])

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('vect', ...), ('clf', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing <metadata_routing>`.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
Name,Type,Value
"classes_ classes_: ndarray of shape (n_classes,)The classes labels. Only exist if the last step of the pipeline is aclassifier.","ndarray[object](18,)","['claude-fable-5','claude-opus-4-6','claude-opus-4-8',...,'gpt-6-astra', 'kimi-k3','qwen3.8-max']"
,"transformer_list transformer_list: list of (str, transformer) tuplesList of transformer objects to be applied to the data. The firsthalf of each tuple is the name of the transformer. The transformer canbe 'drop' for it to be ignored or can be 'passthrough' for features tobe passed unchanged... versionadded:: 1.1 Added the option `""passthrough""`... versionchanged:: 0.22 Deprecated `None` as a transformer in favor of 'drop'.","[('palavras', ...), ('caracteres', ...)]"
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary <n_jobs>`for more details... versionchanged:: v0.20 `n_jobs` default changed from 1 to None",None
,"transformer_weights transformer_weights: dict, default=NoneMultiplicative weights for features per transformer.Keys are transformer names, values the weights.Raises ValueError if key not present in ``transformer_list``.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each transformer will beprinted as it is completed.",False
,"verbose_feature_names_out verbose_feature_names_out: bool, default=TrueIf True, :meth:`get_feature_names_out` will prefix all feature nameswith the name of the transformer that generated that feature.If False, :meth:`get_feature_names_out` will not prefix any featurenames and will error if feature names are not unique... versionadded:: 1.5",True


## 7.2 Submissão

In [3]:
teste = pd.read_csv(DATA_DIR / "test.csv")
submissao = pd.DataFrame({"id": teste["id"], "model": modelo.predict(teste["text"])})

SAIDA_DIR.mkdir(parents=True, exist_ok=True)
submissao.to_parquet(SAIDA_DIR / "submission.parquet", index=False)
print(f"submission.parquet: {len(submissao)} linhas")
submissao.head()

submission.parquet: 3 linhas


,id,model
0,00353fb1c541,gemini-3.6-flash
1,005f5a9a6139,gpt-5.6-terra
2,0062a53d5487,gemini-3.7-flash
